<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 자기지도 학습 · 멀티모달 · 00. CPC: 잠재공간에서 미래 예측

## Representation Learning with Contrastive Predictive Coding

- **원 논문:** [Representation Learning with Contrastive Predictive Coding](https://arxiv.org/abs/1807.03748)
- **저자 / 발표:** Aaron van den Oord, Yazhe Li, Oriol Vinyals · arXiv (2018)
- **난이도 / 예상 시간:** 중급 · 약 65분
- **선수 지식:** GRU, mutual information, contrastive learning
- **로컬 학습 데이터:** `data/field_curriculum/multimodal_pairs.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** encoder와 autoregressive context로 미래 latent를 batch negatives 중 식별하는 InfoNCE를 학습합니다.

**축소하거나 생략한 부분:** speech·image·text·RL 대규모 실험 대신 240개 길이-8 합성 시퀀스를 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2.1, Eq. (1) | 관측 x_t→latent z_t encoder | latent shape |
| §2.1, Eq. (2) | 과거 latent→context c_t | 미래 누설 없는 prefix |
| §2.2, Eq. (3) | bilinear density-ratio score | N×N logits |
| §2.3, Eq. (4) 및 Figure 1 | InfoNCE로 positive future 식별 | loss·top-1 |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** sequence [B,T,D] → context/future [B,H] → contrastive logits [B,B]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(17)

DATA_PATH = Path("data/field_curriculum/multimodal_pairs.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
images = torch.tensor(raw["images"], dtype=torch.float32)
tokens = torch.tensor(raw["tokens"], dtype=torch.long)
class_ids = torch.tensor(raw["class_ids"], dtype=torch.long)
sequences = torch.tensor(raw["sequences"], dtype=torch.float32)
train_idx = torch.tensor(raw["train_idx"], dtype=torch.long)
test_idx = torch.tensor(raw["test_idx"], dtype=torch.long)
assert len(images) == len(tokens) == len(class_ids) == len(sequences)
assert images.shape[1:] == (1, 8, 8) and tokens.ndim == 2 and sequences.ndim == 3
print(
    f"pairs={len(images)}, classes={class_ids.unique().numel()}, "
    f"image={tuple(images.shape[1:])}, tokens={tokens.shape[1]}"
)
print(ACCELERATOR.summary())
print("dataset/index preprocessing stays on CPU, while neural training uses DEVICE")

## 포트폴리오 해설: 표현, 목적함수, 업데이트

$$\mathcal L_{N}=-\mathbb E\!\left[\log \frac{f_k(x_{t+k},c_t)}{\sum_{x_j\in X}f_k(x_j,c_t)}
\right]$$

- **기호 정의:** c_t는 과거 context, x_{t+k}는 positive 미래, X는 batch 후보 집합입니다.
- **수식의 역할:** 미래 latent를 batch negative 중 식별해 관측 간 공유 정보를 보존합니다.
- **구현 이유:** representation 붕괴·modality 누설·잘못된 positive 연결을
  코드에서 확인할 수 있도록 핵심 수식을 view 준비와 loss 경계로 분리했습니다.
- **Task/코드 대응:** Task P의 `prepare_views`, 논문 전용 `nn.Module`의
  `forward`, `compute_loss`, `training_step`이 §2.1, Eq. (1)을 구현합니다.
- **입출력 shape:** 시퀀스 [B, T, 4] → latent/context [B, T, D] → score [B, B]
- **평가:** InfoNCE 감소와 미래 식별 top-1 accuracy
- **원문 대비 한계:** speech·image·text·RL 대규모 실험 대신 240개 길이-8 합성 시퀀스를 사용합니다.

구현은 `로컬 데이터 통계 → view/modality 준비 → model → objective/update → 평가`
순서입니다. 이 순서는 데이터 누설과 stop-gradient 경계를 코드에서 찾기 쉽게 합니다.

## 1. Figure 1: `x → z → c → 미래 z`
같은 batch의 다른 시퀀스 미래 latent가 negative 역할을 합니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2.1, Eq. (1) / §2.1, Eq. (2)
- **이 셀의 책임:** 관측 x_t→latent z_t encoder / 과거 latent→context c_t
- **Tensor shape 계약:** sequence [B,T,D] → context/future [B,H] → contrastive logits [B,B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** latent shape / 미래 누설 없는 prefix. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 00-1: Eq. (1) encoder와 Eq. (2) autoregressive context를 정의하세요.
class CPCModel(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, input_dim=4, latent=12):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def encode_context(self, x: Tensor, time: int):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2.2, Eq. (3) / §2.3, Eq. (4) 및 Figure 1
- **이 셀의 책임:** bilinear density-ratio score / InfoNCE로 positive future 식별
- **Tensor shape 계약:** sequence [B,T,D] → context/future [B,H] → contrastive logits [B,B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** N×N logits / loss·top-1. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 00-2: Eq. (3)–(4)의 in-batch InfoNCE를 구현하세요.
def cpc_loss(model: nn.Module, batch: Tensor, time=3, horizon=1):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §2.3, Eq. (4) 및 Figure 1
- **이 셀의 책임:** InfoNCE로 positive future 식별
- **Tensor shape 계약:** sequence [B,T,D] → context/future [B,H] → contrastive logits [B,B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** loss·top-1. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 00-3: InfoNCE를 학습하고 positive-future top-1을 시각화하세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### Task P1 · modality/view 준비와 핵심 모델

image·text·sequence의 shape를 먼저 고정하고 augmentation 또는 modality
경계를 명시한 뒤 논문 전용 `nn.Module`을 구성합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §2.1, Eq. (1) / §2.1, Eq. (2) / §2.2, Eq. (3) / §2.3, Eq. (4) 및 Figure 1
- **이 셀의 책임:** 관측 x_t→latent z_t encoder / 과거 latent→context c_t / bilinear density-ratio score / InfoNCE로 positive future 식별
- **Tensor shape 계약:** sequence [B,T,D] → context/future [B,H] → contrastive logits [B,B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** latent shape / 미래 누설 없는 prefix / N×N logits / loss·top-1. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO P-1: 논문의 modality 준비, 핵심 module, objective와 평가를 구현하세요.
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    hidden_dim: int = 16
    temperature: float = 0.2
    learning_rate: float = 0.02
    steps: int = 14


def prepare_views(batch: Tensor) -> tuple[Tensor, Tensor]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


class CpcPortfolioModel(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, config: PortfolioConfig):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, prefix: Tensor, future: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def compute_loss(self, logits: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def training_step(self, prefix: Tensor, future: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### Task P2 · objective, stop-gradient, update

positive/negative, online/target 또는 masked/visible 경계를 확인하면서
`fit_portfolio_model`의 gradient와 EMA update 순서를 추적합니다.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §2.3, Eq. (4) 및 Figure 1
- **이 셀의 책임:** InfoNCE로 positive future 식별
- **Tensor shape 계약:** sequence [B,T,D] → context/future [B,H] → contrastive logits [B,B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** loss·top-1. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: CpcPortfolioModel, prefix: Tensor, future: Tensor
) -> list[float]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### Task P3 · 표현 품질 평가

학습 loss와 retrieval·reconstruction·conditioning 지표를 분리해 계산합니다.
작은 로컬 pair의 수치는 원 논문의 downstream benchmark와 직접 비교하지 않습니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §2.3, Eq. (4) 및 Figure 1
- **이 셀의 책임:** InfoNCE로 positive future 식별
- **Tensor shape 계약:** sequence [B,T,D] → context/future [B,H] → contrastive logits [B,B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** loss·top-1. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: CpcPortfolioModel, prefix: Tensor, future: Tensor
) -> dict[str, float]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.